In [3]:
import pandas as pd
import re
from nltk.corpus import stopwords
from itertools import combinations
from collections import Counter

df = pd.read_csv('files/processed_posts.csv')

stop_words = set(stopwords.words('english'))
custom_stop = {'rt', 'amp', 'via'}
stop_words |= custom_stop

def clean_tokenize(text):
    text = str(text).lower()
    text = text.replace('\\n', ' ')
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

df['tokens'] = df['full_text'].apply(clean_tokenize)

# --- Phase 5 starts here ---
pair_counts = Counter()
word_counts = Counter()

for tokens in df['tokens']:
    unique_tokens = set(tokens)
    word_counts.update(unique_tokens)
    for pair in combinations(sorted(unique_tokens), 2):
        pair_counts[pair] += 1

print(pair_counts.most_common(20))

[(('bank', 'west'), 1067), (('bank', 'gaza'), 421), (('gaza', 'west'), 421), (('bank', 'israeli'), 403), (('israeli', 'west'), 403), (('bank', 'israel'), 395), (('israel', 'west'), 394), (('bank', 'palestinian'), 321), (('palestinian', 'west'), 321), (('gaza', 'israel'), 219), (('bank', 'occupied'), 212), (('occupied', 'west'), 212), (('bank', 'palestinians'), 208), (('palestinians', 'west'), 208), (('israeli', 'palestinian'), 184), (('bank', 'people'), 174), (('people', 'west'), 174), (('bank', 'palestine'), 162), (('palestine', 'west'), 162), (('bank', 'one'), 156)]


In [4]:
import math

N = len(df)  # total posts, used as the basis for probability

def calculate_pmi(word1, word2, pair_count, word_counts, N):
    p_word1 = word_counts[word1] / N
    p_word2 = word_counts[word2] / N
    p_pair = pair_count / N
    return math.log2(p_pair / (p_word1 * p_word2))

# minimum pair count filter — avoids PMI being dominated by rare noisy pairs
# (two words that each appear once, together, get a misleadingly huge PMI score)
MIN_PAIR_COUNT = 10

pmi_scores = {}
for pair, count in pair_counts.items():
    if count >= MIN_PAIR_COUNT:
        word1, word2 = pair
        pmi_scores[pair] = calculate_pmi(word1, word2, count, word_counts, N)

top_pmi = sorted(pmi_scores.items(), key=lambda x: x[1], reverse=True)[:20]
for pair, score in top_pmi:
    print(pair, round(score, 2))

('gvir', 'itamar') 6.06
('hormuz', 'strait') 6.04
('compounded', 'overlooked') 5.98
('overlooked', 'unfold') 5.98
('drones', 'missiles') 5.93
('compounded', 'devastation') 5.88
('ben', 'gvir') 5.82
('ben', 'itamar') 5.82
('compounded', 'pain') 5.74
('seeing', 'unfold') 5.74
('devastation', 'overlooked') 5.63
('compounded', 'seeing') 5.63
('broken', 'unfold') 5.6
('broken', 'compounded') 5.49
('centuries', 'compounded') 5.48
('drone', 'missile') 5.48
('systemic', 'unfold') 5.48
('devastation', 'loss') 5.47
('overlooked', 'pain') 5.46
('compounded', 'loss') 5.44


In [5]:
df[df['tokens'].apply(lambda t: 'compounded' in t and 'overlooked' in t)]['full_text'].tolist()

['Ed Sheeran walked onstage in Philadelphia Saturday night looking like a man who hadn’t slept in a week and then opened his first show since the Macklemore fallout by saying the quiet part out loud.\\n\\n“I’ve had to take and make difficult decisions, and I am making mistakes, and I’m so, so sorry.”\\n\\nThat’s how the Loop Tour’s first post-controversy night began. No openers. No backing band. Just Sheeran, a guitar, and a crowd that had watched the entire supporting bill walk away after Macklemore was pulled from the tour over his onstage “Free Palestine” remarks.\\n\\nHe didn’t dodge the politics. He called the October 7 attacks “horrific” and said they compounded centuries of Jewish pain. In the next breath he called what’s happening in Gaza “catastrophic, unjustifiable and disproportionate,” and said the situation in the West Bank cannot be overlooked. Then he drew a line: his concerts are supposed to be about unity, not a political forum. He also pushed back on venues pre-approv

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Vectorize the cleaned token lists (joined back to strings for TF-IDF)
df['tokens_str'] = df['tokens'].apply(lambda t: ' '.join(t))

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['tokens_str'])

# Pairwise similarity — this is an N x N matrix, fine for 1,070 posts
similarity_matrix = cosine_similarity(tfidf_matrix)

SIMILARITY_THRESHOLD = 0.8

# Group posts: for each post, find others above threshold, keep only the first in each group
seen = set()
keep_indices = []

for i in range(len(df)):
    if i in seen:
        continue
    keep_indices.append(i)
    similar = np.where(similarity_matrix[i] >= SIMILARITY_THRESHOLD)[0]
    seen.update(similar)

df_dedup = df.iloc[keep_indices].reset_index(drop=True)

print(f"Before: {len(df)} posts, After: {len(df_dedup)} posts, Removed: {len(df) - len(df_dedup)}")

Before: 1070 posts, After: 1028 posts, Removed: 42


In [7]:
# Sanity check: did the Sheeran cluster actually collapse?
sheeran_count_before = df['tokens'].apply(lambda t: 'sheeran' in t).sum()
sheeran_count_after = df_dedup['tokens'].apply(lambda t: 'sheeran' in t).sum()
print(f"Sheeran posts before: {sheeran_count_before}, after: {sheeran_count_after}")

Sheeran posts before: 76, after: 69


In [11]:
# Known duplicate clusters — add more as we find them
duplicate_clusters = [
    {
        'name': 'sheeran_statement',
        'match_phrase': 'compounded centuries of Jewish pain'
    }
]

drop_indices = []
for cluster in duplicate_clusters:
    matches = df[df['full_text'].str.contains(cluster['match_phrase'], case=False, na=False)]
    indices = matches.index.tolist()
    keep = indices[0]          # keep first occurrence
    drop = indices[1:]         # drop the rest
    drop_indices.extend(drop)
    print(f"{cluster['name']}: {len(indices)} posts found, keeping 1, dropping {len(drop)}")

df_dedup = df.drop(index=drop_indices).reset_index(drop=True)
print(f"\nBefore: {len(df)} posts, After: {len(df_dedup)} posts, Removed: {len(drop_indices)}")

sheeran_statement: 13 posts found, keeping 1, dropping 12

Before: 1070 posts, After: 1058 posts, Removed: 12


In [12]:
pair_counts_dedup = Counter()
word_counts_dedup = Counter()

for tokens in df_dedup['tokens']:
    unique_tokens = set(tokens)
    word_counts_dedup.update(unique_tokens)
    for pair in combinations(sorted(unique_tokens), 2):
        pair_counts_dedup[pair] += 1

N_dedup = len(df_dedup)
MIN_PAIR_COUNT = 10

pmi_scores_dedup = {}
for pair, count in pair_counts_dedup.items():
    if count >= MIN_PAIR_COUNT:
        word1, word2 = pair
        pmi_scores_dedup[pair] = calculate_pmi(word1, word2, count, word_counts_dedup, N_dedup)

top_pmi_dedup = sorted(pmi_scores_dedup.items(), key=lambda x: x[1], reverse=True)[:20]
for pair, score in top_pmi_dedup:
    print(pair, round(score, 2))

('gvir', 'itamar') 6.05
('hormuz', 'strait') 6.02
('drones', 'missiles') 5.91
('ben', 'gvir') 5.8
('ben', 'itamar') 5.8
('disproportionate', 'unjustifiable') 5.79
('lineup', 'rapper') 5.73
('drone', 'missile') 5.46
('catastrophic', 'unjustifiable') 5.38
('horrific', 'unjustifiable') 5.37
('catastrophic', 'disproportionate') 5.37
('kippur', 'yom') 5.29
('philadelphia', 'unjustifiable') 5.21
('arabia', 'riyadh') 5.21
('drone', 'missiles') 5.2
('kraft', 'robert') 5.15
('catastrophic', 'philadelphia') 5.15
('arabia', 'saudi') 5.14
('kraft', 'owners') 5.14
('owners', 'robert') 5.14
